# Make results tibble

This script makes `results_tibble.csv` and `estimate_tibble.csv`

In [ ]:

library(tidyverse)


-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.5.1
v ggplot2   4.0.0     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.0.4     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

In [ ]:
d_0 <- d_0 |> 
  bind_rows(subset(d_20_covs, b_x == 0)) |> 
  bind_rows(subset(d_n50, b_x == 0)) |> 
  bind_rows(subset(d_wo_x_0, b_x == 0)) |> 
  bind_rows(subset(d_wo_x_1, b_x == 0)) |> 
  bind_rows(d2_0)

d_05 <- d_05 |> 
  bind_rows(subset(d_20_covs, b_x == 0.5)) |> 
  bind_rows(subset(d_n50, b_x == 0.5)) |> 
  bind_rows(subset(d_wo_x_0, b_x == 0.5)) |> 
  bind_rows(subset(d_wo_x_1, b_x == 0.5)) |> 
  bind_rows(d2_05)
  
d_02 <- d_02_0 |> 
  bind_rows(d_02_1)


rm(d_20_covs, d_n50, d2_0, d2_05, d_02_0, d_02_1, d_wo_x_0, d_wo_x_1)


Combine and save out combined data

In [ ]:
d_all <- d_0 |> 
  rbind(d_02, d_05) |> 
  glimpse()


Rows: 194,400,000
Columns: 16
$ job_num       <int> 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 10~
$ method        <chr> "no_covs", "all_covs", "p_hacked", "r", "partial_r", "fu~
$ simulation_id <int> 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3,~
$ estimate      <dbl> 0.21582038, 0.18719362, 0.22183271, 0.18118454, 0.181184~
$ SE            <dbl> 0.1567998, 0.1514230, 0.1565812, 0.1514937, 0.1514937, 0~
$ p_value       <dbl> 0.1707744, 0.2183858, 0.1586932, 0.2336270, 0.2336270, 0~
$ ndf           <int> 1, 5, 3, 2, 2, 2, 4, 1, 5, 4, 2, 2, 2, 3, 1, 5, 1, 2, 2,~
$ ddf           <int> 148, 144, 146, 147, 147, 147, 145, 148, 144, 145, 147, 1~
$ covs_tpr      <dbl> 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,~
$ covs_fpr      <dbl> 0.0000000, 1.0000000, 0.6666667, 0.0000000, 0.0000000, 0~
$ n_obs         <int> 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 1~
$ b_x           <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,~
$ n_covs  

### Calculate type 1 and type 2 error

Calculate error rates

In [ ]:
d_tibble <- d_all |> 
  group_by(method, n_obs, n_covs, r_ycov, p_good_covs, b_x) |> 
  summarise(type_I = mean(p_value < 0.05),
            type_II = mean(p_value >= 0.05),
            .groups = "drop")

d_tibble <- d_tibble|> 
  mutate(type_I = if_else(b_x != 0.0, NA_real_, type_I),
         type_II = if_else(b_x == 0.0, NA_real_, type_II))


Save out tibble

In [ ]:
d_tibble |> 
  write_csv(here::here(path_processed, "results_tibble.csv"))


### Create Parameter Estimate summary table

In [ ]:
d_param <- d_all |> 
  group_by(method, b_x) |> 
  summarise(mean = mean(estimate), .groups = "drop") |> 
  pivot_wider(names_from = b_x, values_from = mean)  |> 
  glimpse()


Rows: 9
Columns: 4
$ method <chr> "all_covs", "full_lm", "full_lm_wo_x", "lasso", "lasso_wo_x", "~
$ `0`    <dbl> -8.416024e-05, -3.630614e-05, -4.612105e-05, -4.475049e-05, -6.~
$ `0.2`  <dbl> 0.2000111, 0.1999958, 0.1915533, 0.2000109, 0.1946056, 0.199983~
$ `0.5`  <dbl> 0.4999267, 0.4999396, 0.4793405, 0.4999455, 0.4869628, 0.500003~